In [12]:
import requests
import pandas as pd
import json
import os

In [13]:
opt_code = ('')
                            # action(1/6)：从ifind super command勾选复制**同一到期日的合约**，粘贴到此
opt_code = (
    '10011905.SH,10011906.SH,10011907.SH,10011908.SH,10011909.SH,10011910.SH,10011911.SH,10011912.SH,10011913.SH,10011914.SH,10011916.SH,10011915.SH,10011917.SH,10011918.SH,10011919.SH,10011920.SH,10011921.SH,10011922.SH,10011947.SH,10011948.SH,10011949.SH,10011950.SH,10011973.SH,10011974.SH,10012007.SH,10012008.SH,10012009.SH,10012010.SH,10012107.SH,10012108.SH,10012121.SH,10012122.SH,10012123.SH,10012124.SH,10012151.SH,10012152.SH,10012153.SH,10012154.SH,10012155.SH,10012156.SH,10012286.SH,10012285.SH,10012287.SH,10012288.SH,10012289.SH,10012290.SH'
    )

underlying = '588000.SH'    # action(2/6)：输入合约标的指数的代码

opt_date = '2026-07-30'     # action(3/6)：输入查询的日期，最好填最近一个交易日以获取最新的期权链

opt_EX = '2608'             # action(4/6)：输入合约月份

                            # action(5/6): 更新ifind token，刷新后access token有效期为7天
accessToken = '66c52246ec82ec4134c216353e415e739ac84584.signs_NzUzNDI0OTM5'

                            # action(6/6): 运行全部单元格，查看底部状态是否为“成功更新”

---
---


In [14]:
opt_var = ['ths_option_short_name_option']
start_date = opt_date
end_date = start_date
underlying_code = str(underlying + '_' + opt_EX)

In [15]:
RT_Url = 'https://quantapi.51ifind.com/api/v1/date_sequence'
thsHeaders = {"Content-Type":"application/json","access_token":accessToken}

thsPara_opt = {"codes":opt_code,
               'startdate':start_date,
               'enddate': end_date,
               'indipara': [{"indicator": x, "indiparams": []} for x in opt_var]
               }

thsResponse = requests.post(url=RT_Url, json=thsPara_opt, headers=thsHeaders)
data_opt = thsResponse.json()



In [16]:
rows = []
for item in data_opt["tables"]:
    thscode = item["thscode"]
    time = item["time"][0]  # time 是列表，取第一个元素
    name = item["table"]["ths_option_short_name_option"][0]  # 名称也是列表，取第一个

    # 拼成一行
    rows.append({
        "thscode": thscode,
        'date': time,
        'name': name
    })

df = pd.DataFrame(rows)

# 基础检查:上交所规定合约购/沽成对挂牌，df有偶数行
print(df)
if len(df) % 2 != 0:
    raise Exception("期权购/沽数不一致，请检查opt_code输入！")


        thscode        date         name
0   10011905.SH  2026-07-30  科创50购8月1900
1   10011906.SH  2026-07-30  科创50购8月1950
2   10011907.SH  2026-07-30  科创50购8月2000
3   10011908.SH  2026-07-30  科创50购8月2050
4   10011909.SH  2026-07-30  科创50购8月2100
5   10011910.SH  2026-07-30  科创50购8月2150
6   10011911.SH  2026-07-30  科创50购8月2200
7   10011912.SH  2026-07-30  科创50购8月2250
8   10011913.SH  2026-07-30  科创50购8月2300
9   10011914.SH  2026-07-30  科创50沽8月1900
10  10011916.SH  2026-07-30  科创50沽8月2000
11  10011915.SH  2026-07-30  科创50沽8月1950
12  10011917.SH  2026-07-30  科创50沽8月2050
13  10011918.SH  2026-07-30  科创50沽8月2100
14  10011919.SH  2026-07-30  科创50沽8月2150
15  10011920.SH  2026-07-30  科创50沽8月2200
16  10011921.SH  2026-07-30  科创50沽8月2250
17  10011922.SH  2026-07-30  科创50沽8月2300
18  10011947.SH  2026-07-30  科创50购8月2350
19  10011948.SH  2026-07-30  科创50购8月2400
20  10011949.SH  2026-07-30  科创50沽8月2350
21  10011950.SH  2026-07-30  科创50沽8月2400
22  10011973.SH  2026-07-30  科创50购8月2450
23  10011974.SH 

In [17]:
# 可保存为其他格式的临时文件以便检查
# df.to_csv("thscode_name.csv", index=False, encoding="utf-8-sig")
# df.to_excel("thscode_name.xlsx", index=False)


In [18]:
file_path = "option_code_map.json"
# ==================================================
# 1. 从 df 生成最新的 code_to_name 字典
new_code_map = df.set_index("thscode")["name"].to_dict()

# 2. 如果文件存在，读取原有数据；不存在则创建空字典
if os.path.exists(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        all_data = json.load(f)
else:
    all_data = {}

# 3. 【只更新目标合约，其他合约完全不动】
all_data[underlying_code] = {
    "code_to_name": new_code_map
}

# 4. 写回文件（覆盖整个文件，但只改了目标合约）
with open(file_path, "w", encoding="utf-8") as f:
    json.dump(all_data, f, ensure_ascii=False, indent=2)

print(f"✅ 成功更新 {underlying_code} 类 code_to_name")

✅ 成功更新 588000.SH_2608 类 code_to_name
